# The variable you almost overlooked

**A guided investigation · Chapter 6 · 30–45 minutes · No Python experience needed**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmg777/intro2causal/blob/main/notebooks_colab/explore-synthetic_ovb.ipynb)

> **The question**  
> How can ability complicate the relationship between schooling and earnings?

Real data rarely reveal every reason that people choose more education. This teaching simulation gives us an advantage: the data-generating code includes ability, schooling, occupation, and log earnings, so we can inspect relationships that might be hidden in practice. Each row is a simulated person. Ability influences both schooling and earnings, while occupation is partly a consequence of schooling and provides another route to earnings. These roles are different: a variable that precedes schooling is not interchangeable with one that follows it. We will start with the familiar schooling–earnings pattern, color observations by ability, and compare it with the schooling–occupation relationship. The numbers are generated for learning, not evidence about an actual population. Their value is that they let us reason explicitly about why adding a variable to an analysis can change what question we are answering.

### By the end, you will be able to…

- Visualize a confounder and distinguish it from a variable that lies on a causal pathway.
- Read and modify three interactive charts using short Python examples.
- Explain one reason a descriptive pattern alone cannot settle this case’s causal question.

### Before you see the evidence

If ability raises both schooling and earnings, would the raw schooling–earnings pattern exaggerate the effect of schooling?

Write two sentences now. You will return to them at the end.

**Your route:** meet the data → inspect three views → modify and try → ask the causal question.

## Your first few minutes in Colab · 3 minutes

You do not need prior Python experience. A **cell** is one block of text or code. Read the text, then press the **▶** beside the next code cell (or Shift+Enter). Its output appears underneath. Run cells in order: later cells use tables created earlier.

Choose **File → Save a copy in Drive** if you want to keep your changes. If a name is “not defined,” run the earlier setup cells again. **Runtime → Restart session and run all** gives you a fresh start. An error does not mean you have damaged the data: this notebook only reads it.

Keep a short learning log in a text cell. Before each chart, make a prediction. Afterward, write one observation with a number and one question that the chart cannot answer.

**Chart controls:** hover for values, click legend labels to hide/show a group, drag to zoom, and double-click to reset. A chart’s menu also offers a reset/autoscale control.

The setup installs Plotly Express if needed. `import` makes a library available; `pd` and `px` are short names for pandas (tables) and Plotly Express (charts).

In [ ]:
%pip install -q "plotly>=6,<7"

In [ ]:
import pandas as pd
import plotly.express as px
from IPython.display import display

# One consistent chart style; no plotting experience needed.
px.defaults.template = "plotly_white"
px.defaults.color_continuous_scale = "Blues"
px.defaults.color_discrete_sequence = ["#2563eb", "#7c3aed", "#059669", "#64748b"]

## Meet the data · 5 minutes

**One row:** Simulated individual.  
**Stored size:** 2,000 rows × 4 columns.

| Column | What it records |
|---|---|
| `schooling` | Simulated completed years of schooling, clipped to 6–20 and rounded. |
| `earnings` | Simulated log earnings; the column name does not indicate dollars. |
| `ability` | Simulated pre-existing ability score (standard normal in Chapter 6). |
| `occupation` | Simulated occupation-status category 1–5, partly caused by schooling; a mediator. |

The column earnings contains simulated log earnings. Occupation is generated after schooling and acts as a mediator. The generator includes clipping and rounding, so simplistic exact-effect claims are inappropriate.

`read_csv` downloads a table. We call it `df`. `head(5)` shows just five rows, not a separate sample used for the analysis. Square brackets select columns by their names.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/cmg777/intro2causal/main/data/ch6/synthetic_ovb.csv"
df = pd.read_csv(DATA_URL)
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns")
display(df[['schooling', 'earnings', 'ability', 'occupation']].head(5))

### Pause: what do the rows mean?

Choose one displayed row. Translate it into a sentence using the variable guide. Which values are categories or identifiers? Which are quantities? A numeric storage type does not necessarily make a code a meaningful measurement.

The next table checks types, missing values, and the number of distinct values for the key columns. `isna()` marks missing cells; `sum()` counts them. Missing is not zero. Ask which comparison would lose a row if one of its required values were missing.

In [ ]:
key_columns = ['schooling', 'earnings', 'ability', 'occupation']
checks = pd.DataFrame({"storage_type": df[key_columns].dtypes.astype(str),
                       "missing_rows": df[key_columns].isna().sum(),
                       "distinct_values": df[key_columns].nunique()})
display(checks)
print("The charts below omit missing values required for each plotted quantity.")

### A small summary before plotting

`describe()` summarizes numeric columns: `count` counts available values, `mean` is the average, and `50%` is the median. The median splits ordered values into two halves. Averages of identifiers or arbitrary category codes are not useful descriptions, even if Python calculates them.

For category-like key variables, the second table counts values, including missing entries. Check these labels before interpreting a chart.

In [ ]:
numeric_keys = df[key_columns].select_dtypes(include="number").columns
if len(numeric_keys):
    display(df[numeric_keys].describe().round(2))
for column in key_columns:
    if df[column].nunique() <= 8:
        display(df[column].value_counts(dropna=False).rename("Rows").to_frame())

## Read the chart code · 2 minutes

`px.bar`, `px.scatter`, `px.line`, and `px.box` choose a chart type. `x` and `y` name columns for the axes; `color` distinguishes groups or values. `labels` translates column names for readers. `fig.show()` displays the result.

A table transformation and a plot answer different questions: first decide what each plotted row represents, then choose how to display it. Before running each chart, predict what you expect to see.

In [ ]:
# Display labels: run once; you do not need to memorize this mapping.
labels = {'ability': 'Simulated ability score',
 'ability_group': 'Ability group',
 'census': 'Census year',
 'earnings': 'Log earnings',
 'group': 'State group',
 'occupation': 'Occupation status (1–5)',
 'offer': 'Scholarship assignment',
 'plan': 'Assigned insurance plan',
 'schooling': 'Years of schooling',
 'state_name': 'Simulated state'}

## Visual investigation 1 · See the tempting association

**Predict:** Do simulated people with more schooling earn more?

The generated earnings column is on a logarithmic scale. Transparency reveals overlapping observations at discrete schooling levels.

In [ ]:
fig = px.scatter(df, x="schooling", y="earnings", opacity=0.4, labels=labels, title="A pattern that needs an explanation")
fig.show()

**Explore the chart:** Inspect variation among people with the same schooling. A strong overall pattern still leaves individual differences.

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

The association includes both schooling pathways and ability-related selection. The generator gives us a way to examine those components conceptually.

</details>

## Visual investigation 2 · Reveal the common influence

**Predict:** Where do higher-ability people sit in the same chart?

Color now represents a continuous ability score. A common influence on both axes can create or steepen an association.

In [ ]:
fig = px.scatter(df, x="schooling", y="earnings", color="ability", opacity=0.6, labels=labels, title="Ability is related to both axes")
fig.show()

**Explore the chart:** Hover over different color regions. Does ability appear related to schooling, earnings, or both?

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

This is the visual signature of the confounding built into the simulation. Coloring points reveals a relationship; it does not statistically remove it.

</details>

## Visual investigation 3 · A consequence is a different kind of variable

**Predict:** How does occupation vary with schooling?

Occupation is generated partly from schooling. We average its coded status level by schooling to see this downstream relationship.

In [ ]:
jobs = df.groupby("schooling", as_index=False)["occupation"].mean()
fig = px.line(jobs, x="schooling", y="occupation", markers=True, labels=labels, title="Schooling also changes a pathway to earnings")
fig.show()

**Explore the chart:** Follow the line from low to high schooling. Why is occupation different from pre-existing ability?

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

Holding a mediator fixed can remove part of a total effect. The right adjustment depends on the causal question, not simply on adding more columns.

</details>

## Make it yours · 8 minutes

### 1. Modify an example

Change the color variable in the earnings scatter from ability to occupation. Explain why the causal role of color has changed.

Copy that chart cell before editing, so you can compare the original and modified outputs. Say what changed in your question as well as in your code.

### 2. Try a small investigation

Plot mean log earnings by schooling separately for people above and below the median ability score.

Write a prediction first. The optional starter cell below is safe to leave unchanged; later cells do not depend on your answer.

In [ ]:
# Your investigation goes here.
# Describe your expected pattern before writing code.

<details>
<summary>Hint 1 · choose the right evidence</summary>

Create an ability_group column using a comparison with the median.

</details>

<details>
<summary>Hint 2 · a Python starting point</summary>

Group by both schooling and ability_group, then draw colored lines.

</details>

<details>
<summary>Optional worked solution · open after trying</summary>

<pre><code>groups = df.copy()
groups[&quot;ability_group&quot;] = (groups[&quot;ability&quot;] &gt;= groups[&quot;ability&quot;].median()).map({True: &quot;Higher ability&quot;, False: &quot;Lower ability&quot;})
means = groups.groupby([&quot;schooling&quot;, &quot;ability_group&quot;], as_index=False)[&quot;earnings&quot;].mean()
fig = px.line(means, x=&quot;schooling&quot;, y=&quot;earnings&quot;, color=&quot;ability_group&quot;, markers=True, labels=labels)
fig.show()</code></pre>

The broad ability split helps visualize confounding but does not fully hold continuous ability constant. It is not a controlled causal estimate.

Copy the code into the starter cell to run it.

</details>

## What can you now say? · 4 minutes

Return to your initial prediction. Write a four-sentence evidence note:

1. **I expected…** State your prediction.
2. **I observed…** Use a chart, a comparison, and the correct units.
3. **This does not establish…** Name a causal claim the evidence cannot settle.
4. **To learn more, I would need…** Propose an additional measurement or a more credible comparison.

### Carry this question into the chapter

**Why might controlling for occupation remove part of the schooling effect you wanted to learn about?**

[Continue with Chapter 6: The Wages of Schooling](https://cmg777.github.io/intro2causal/book/_book/notebooks_quarto/06-wages-of-schooling.html#seeing-ovb-in-action) · [Return to the case collection](https://cmg777.github.io/intro2causal/data-dictionary.html#ch6-synthetic-ovb-csv)

### Source and preparation notes

[Repository teaching simulation, code/python/ch6/generate_synthetic.py](https://github.com/cmg777/intro2causal/blob/main/code/python/ch6/generate_synthetic.py)

Generated by code/python/ch6/generate_synthetic.py (seed 2024, legacy NumPy global RNG). See generation details in source. The generator draws ability, schooling noise, occupation noise, then earnings noise; schooling and occupation are clipped and rounded.

These exploratory charts are based on the committed CSV. They are not a complete replication of the original study. Keep the source study, this prepared extract, and your descriptive results distinct.